### 0. 开始

In [1]:
%load_ext autoreload
%autoreload 2

In [26]:
import os
import utils_z
import cityjson_parser_lod1 as cjpar


In [3]:
conn = utils_z.get_conn("Test20260413", "postgres", "we6666", "localhost", "5432")

In [4]:
city_name = "hamburg"
city_prefix  = "DE_HH"

### 1. 处理 lod 1 CityGML 数据并导入数据库

#### 1.1 批量转换为json

In [ ]:
citygml_tools_path = r"E:\0_mylib\citygml-tools-2.4.1\citygml-tools.bat"

# 先批量转换XML到CityJSON
lod1_xml_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_gml"
lod1_json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_json"
os.makedirs(lod1_json_dir, exist_ok=True)

In [ ]:
# xml_files = [f for f in os.listdir(lod1_xml_dir) if f.endswith(".xml")]
xml_files = [f for f in os.listdir(lod1_xml_dir) if f.endswith(".gml")]
print(f"共{len(xml_files)}个文件待转换")

errors = [] 
for i, filename in enumerate(xml_files):
    input_path = os.path.join(lod1_xml_dir, filename)
    cmd = f'"{citygml_tools_path}" to-cityjson --output="{lod1_json_dir}" "{input_path}"'
    try:
        utils_z.run_cmd(cmd, False)
        if (i + 1) % 50 == 0:
            print(f"转换进度：{i + 1}/{len(xml_files)}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"转换错误：{filename} -> {e}")

print(f"转换完成，失败{len(errors)}个")

#### 1.2 检查数据，确定数据质量、srid

In [5]:
import json

test_json_path = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_json\LoD1_32_551_5935_1_HH.json"

In [6]:
with open(test_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 第一段：顶层结构和对象类型统计
print("顶层keys:", data.keys())

object_counts = {}
for obj_id, obj in data["CityObjects"].items():
    obj_type = obj["type"]
    object_counts[obj_type] = object_counts.get(obj_type, 0) + 1
print("\nCityObject类型统计:")
for k, v in sorted(object_counts.items()):
    print(f"  {k}: {v}")

顶层keys: dict_keys(['type', 'version', 'CityObjects', 'transform', 'vertices', 'metadata'])

CityObject类型统计:
  Building: 240


In [7]:
# 第二段：坐标系和transform
print("\nmetadata:", data.get("metadata", {}))
print("transform:", data["transform"])
print("第一个顶点原始值:", data["vertices"][0])


metadata: {'geographicalExtent': [551000.0, 5935000.0, 5.472, 552000.0, 5936000.0, 95.489]}
transform: {'scale': [0.001, 0.001, 0.001], 'translate': [551014.943, 5934996.673, 5.472]}
第一个顶点原始值: [961564, 935671, 67797]


In [ ]:
# 第三段：第一个Building/BuildingPart的属性和几何

for obj_id, obj in data["CityObjects"].items():
    if obj["type"] in ("Building", "BuildingPart"):
        print(f"\n第一个对象ID: {obj_id}")
        print(f"类型: {obj['type']}")
        print(f"属性: {json.dumps(obj.get('attributes', {}), indent=2, ensure_ascii=False)}")
        for geom in obj.get("geometry", []):
            print(f"几何LOD: {geom.get('lod')}, type: {geom.get('type')}")
                  
        break



第一个对象ID: DEHHALKA5NG0009a
类型: Building
属性: {
  "creationDate": "2023-02-23T00:00:00+08:00",
  "DatenquelleLage": "1000",
  "DatenquelleBodenhoehe": "1100",
  "Gemeindeschluessel": "02200223",
  "DatenquelleDachhoehe": "1000",
  "DatenquelleGeschossanzahl": "1000",
  "Geometrietyp2DReferenz": "3000",
  "Grundrissaktualitaet": "2023-01-26",
  "BezugspunktDach": "2000",
  "measuredHeight": 10.418,
  "function": "31001_1010",
  "storeysAboveGround": 3
}
几何LOD: 1, type: Solid, semantics surfaces数量: 0


In [10]:
# 看LOD分布
lod_counts = {}
for obj_id, obj in data["CityObjects"].items():
    for geom in obj.get("geometry", []):
        lod = str(geom.get("lod"))
        lod_counts[lod] = lod_counts.get(lod, 0) + 1
print("LOD分布:", lod_counts)

LOD分布: {'1': 240}


#### 1.3 变量

In [16]:
block_table_name = f"blocks.{city_name}_blocks"

lod1_table_name_full = f"lod1.{city_name}_buildings_lod1"
lod1_surface_table_name_full = f"lod1.{city_name}_building_surfaces_lod1"

lod1_table_name = f"{city_name}_buildings_lod1"
lod1_surface_table_name = f"{city_name}_building_surfaces_lod1"

target_srid  = 4326
source_srid  = 25832

#### 1.4 建表

In [20]:
# LOD1建筑主表
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod1_table_name_full} (
        building_id     VARCHAR PRIMARY KEY,
        block_id        VARCHAR,
        citygml_id      VARCHAR,
        geom_2d         GEOMETRY(Polygon, {target_srid}),
        height          FLOAT,
        floor_count     INTEGER,
        function        VARCHAR,
        ground_z        FLOAT
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod1_table_name}_geom_idx
    ON {lod1_table_name_full} USING GIST (geom_2d);
""", conn=conn)

# LOD1 surface子表（无citygml_id，面由计算生成）
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod1_surface_table_name_full} (
        surface_id      VARCHAR PRIMARY KEY,
        building_id     VARCHAR REFERENCES {lod1_table_name_full}(building_id),
        surface_type    VARCHAR,
        geom_3d         GEOMETRY(PolygonZ, {target_srid})
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod1_surface_table_name}_building_idx
    ON {lod1_surface_table_name_full} (building_id);
""", conn=conn)

print(city_prefix + " LOD1表创建完成")

DE_HH LOD1表创建完成


#### 1.5 批量入库

In [21]:
lod1_json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_json"

In [22]:
import traceback

In [30]:
json_files = [f for f in os.listdir(lod1_json_dir) if f.endswith(".json")]
print(f"共{len(json_files)}个文件待入库")

total = 0
errors = []

conn.rollback()
# 获取当前最大ID，设置计数器初始值（空表格则为1）
cur = conn.cursor()

cur.execute(f"SELECT MAX(building_id) FROM {lod1_table_name_full}")
max_bid = cur.fetchone()[0]
building_counter = int(max_bid.split("_B_")[1]) + 1 if max_bid else 1

cur.execute(f"SELECT MAX(surface_id) FROM {lod1_surface_table_name_full}")
max_sid = cur.fetchone()[0]
surface_counter = int(max_sid.split("_S_")[1]) + 1 if max_sid else 1

cur.close()

# 遍历json文件，解析并入库，如有一个出错，整个文件跳过
for i, filename in enumerate(json_files):
    filepath = os.path.join(lod1_json_dir, filename)
    try:
        buildings = cjpar.parse_cityjson_lod1(filepath, target_lod="1")
        count, building_counter, surface_counter = cjpar.insert_buildings_lod1(
            buildings, conn,
            lod1_table=lod1_table_name_full,
            surface_table=lod1_surface_table_name_full,
            city_prefix=city_prefix,
            target_srid=target_srid,
            source_srid=source_srid,
            building_counter=building_counter,
            surface_counter=surface_counter
        )
        total += count
        if (i + 1) % 50 == 0:
            print(f"入库进度：{i+1}/{len(json_files)}，已入库建筑：{total}，已入库表面：{surface_counter-1}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")
        traceback.print_exc()

print(f"\n完成！共入库建筑：{total}，共入库表面：{surface_counter-1}")

print(f"失败文件数：{len(errors)}")

共784个文件待入库
入库进度：50/784，已入库建筑：15328，已入库表面：147089
入库进度：100/784，已入库建筑：38867，已入库表面：365384
入库进度：150/784，已入库建筑：68189，已入库表面：643619
入库进度：200/784，已入库建筑：88981，已入库表面：851747
入库进度：250/784，已入库建筑：116545，已入库表面：1156547
入库进度：300/784，已入库建筑：153498，已入库表面：1553888
入库进度：350/784，已入库建筑：179724，已入库表面：1866906
入库进度：400/784，已入库建筑：207030，已入库表面：2154469
入库进度：450/784，已入库建筑：227768，已入库表面：2364684
入库进度：500/784，已入库建筑：254366，已入库表面：2621236
入库进度：550/784，已入库建筑：291165，已入库表面：2965732
入库进度：600/784，已入库建筑：314181，已入库表面：3174638
入库进度：650/784，已入库建筑：340632，已入库表面：3431374
入库进度：700/784，已入库建筑：361536，已入库表面：3619596
入库进度：750/784，已入库建筑：380702，已入库表面：3796396

完成！共入库建筑：384629，共入库表面：3828710
失败文件数：0


### 2 叠合block

In [31]:
# 确认空间索引
utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod1_table_name}_geom_idx
    ON {lod1_table_name_full} USING GIST (geom_2d);
""", conn=conn)
print("开始空间叠合...")

开始空间叠合...


In [32]:
utils_z.run_sql(f"""
    UPDATE {lod1_table_name_full} b
    SET block_id = (
        SELECT bl.block_id
        FROM {block_table_name} bl
        WHERE ST_Within(ST_Centroid(b.geom_2d), bl.geom)
        LIMIT 1
    );
""", conn=conn)
print("空间叠合完成")

空间叠合完成


In [33]:
# 检查匹配情况
result = utils_z.run_sql(f"""
    SELECT
        COUNT(*) AS total,
        COUNT(block_id) AS matched,
        COUNT(*) FILTER (WHERE block_id IS NULL) AS unmatched
    FROM {lod1_table_name_full};
""", conn=conn, fetch=True)

print(f"总建筑数：{result[0][0]}")
print(f"成功匹配block：{result[0][1]}")
print(f"未匹配block：{result[0][2]}")

总建筑数：384629
成功匹配block：354272
未匹配block：30357


In [34]:
# 检查包含LOD1建筑的街区数量
sql_counts = f"SELECT COUNT(DISTINCT block_id) FROM {lod1_table_name_full} WHERE block_id IS NOT NULL;"
(lod1_count,) = utils_z.run_sql(sql_counts, conn=conn, fetch=True)[0]

print(f"包含 LOD1 建筑的街区数量: {lod1_count}")

包含 LOD1 建筑的街区数量: 5157
